# ChessLab: GPU-Accelerated Training & Evaluation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mochiron-desu/chessAI/blob/master/colab/train_gpu.ipynb)

This notebook provides end-to-end GPU training, evaluation, tactical puzzle benchmarking, and interactive inference for the **ChessLab** residual convolutional neural network.

### Quick Start in Google Colab:
1. Switch to GPU: **Runtime** -> **Change runtime type** -> Select **T4 GPU** (or A100/V100).
2. Run all cells sequentially. The first cell below automatically clones the repository and installs all requirements.
3. **Dataset Generation**: To generate custom datasets from 10k up to 100M positions without Google Drive, use [![Open Data Generator](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mochiron-desu/chessAI/blob/master/colab/generate_data.ipynb).

## 1. Auto-Clone Repository & Install Requirements

In [ ]:
# =============================================================================
# STEP 1: Auto-Clone Repo & Configure chesslab Package
# =============================================================================
import os
import sys

# Auto-clone repository if running in Google Colab and files not yet present
if not os.path.exists("src/chesslab"):
    if not os.path.exists("/content/chessAI"):
        print("Cloning repository from https://github.com/mochiron-desu/chessAI.git...")
        !git clone https://github.com/mochiron-desu/chessAI.git /content/chessAI
    if os.path.exists("/content/chessAI"):
        os.chdir("/content/chessAI")
        print(f"Switched working directory to: {os.getcwd()}")

# Install required dependencies & editable package
!pip install -q python-chess zstandard pyyaml tqdm requests scipy
if os.path.exists("pyproject.toml"):
    !pip install -q -e .

# Ensure src/ is on Python search path
for path in [os.path.abspath("."), os.path.abspath("src"), "/content/chessAI", "/content/chessAI/src", "/content/src"]:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

# Verify import
import chesslab
from chesslab.data.dataset import BinaryChessDataset, fast_collate_fn
print(f"SUCCESS: 'chesslab' loaded from: {chesslab.__file__}")

## 2. Hardware Verification & GPU Detection

In [ ]:
# Verify NVIDIA GPU hardware and CUDA support
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU runtime not active. Please select Runtime > Change runtime type > T4 GPU.")

## 3. Dataset Shards Verification & Auto-Unpack / Stream Fallback

Verifies presence of `data/shards/train.bin`. If you downloaded a dataset zip from [generate_data.ipynb](https://colab.research.google.com/github/mochiron-desu/chessAI/blob/master/colab/generate_data.ipynb), upload it to Colab, and this cell will auto-unpack it. If no dataset is present, it automatically streams a quick 10,000-game sample (~30 seconds).

In [ ]:
import os
import glob
import zipfile

# 1. Check for any uploaded dataset zip archive in current directory or /content
zip_candidates = glob.glob("*.zip") + glob.glob("/content/*.zip")
matched_zips = [z for z in zip_candidates if "chesslab" in z.lower() or "data" in z.lower() or "shard" in z.lower()]

if matched_zips and not os.path.exists("data/shards/train.bin"):
    target_zip = matched_zips[0]
    print(f"Found uploaded dataset archive: {target_zip}")
    print("Unpacking archive into workspace...")
    with zipfile.ZipFile(target_zip, "r") as zf:
        zf.extractall(".")
    print("Unpack complete!")

# 2. Check available shards
shards_to_check = [
    "data/shards/train.bin",
    "data/shards/val.bin",
    "data/shards/test.bin",
    "data/shards/train_filtered_25k.bin",
    "data/shards/train_raw_25k.bin",
]

found_shards = {s: os.path.getsize(s) for s in shards_to_check if os.path.exists(s)}

# 3. If train.bin still missing, auto-stream a quick 10,000-game sample on the fly
if not any("train" in k for k in found_shards):
    print("\n[INFO] train.bin not found! Auto-streaming quick sample (10,000 games in ~30s)...")
    from chesslab.data.pipeline import PipelineConfig, run_ingestion_pipeline
    quick_cfg = PipelineConfig(
        source="https://database.lichess.org/standard/lichess_db_standard_rated_2024-01.pgn.zst",
        output_dir="data",
        max_valid_games=10000,
        report_every=2500,
        min_elo=1500,
    )
    run_ingestion_pipeline(quick_cfg)
    print("Quick sample ingestion complete!")
    found_shards = {s: os.path.getsize(s) for s in shards_to_check if os.path.exists(s)}

print("\nReady dataset shards:")
for s, sz in found_shards.items():
    print(f"  [OK] {s} ({sz / (1024*1024):.1f} MB)")

## 4. PyTorch Dataset & High-Throughput GPU DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader
from chesslab.data.dataset import BinaryChessDataset, fast_collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

# Scale batch size for GPU (512 or 1024 for modern GPUs)
BATCH_SIZE = 512 if torch.cuda.is_available() else 128
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# Choose training shard (full train.bin or 25k pilot)
train_shard = "data/shards/train.bin"
if not os.path.exists(train_shard):
    for fallback in ["data/shards/train_filtered_25k.bin", "data/shards/train_raw_25k.bin", "data/shards/test.bin"]:
        if os.path.exists(fallback):
            train_shard = fallback
            break

val_shard = "data/shards/val.bin"

print(f"Loading train shard: {train_shard}")
train_dataset = BinaryChessDataset(train_shard)
print(f"Train dataset contains: {len(train_dataset):,} positions")

val_dataset = BinaryChessDataset(val_shard) if os.path.exists(val_shard) and os.path.getsize(val_shard) > 0 else None
if val_dataset:
    print(f"Validation dataset contains: {len(val_dataset):,} positions")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=fast_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=fast_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
) if val_dataset else None

## 5. Model Architecture (ChessResNet)

In [ ]:
from chesslab.models.resnet import ChessResNet

# Architecture configuration
# 6 blocks, 64 channels = ~960k parameters (baseline)
# 10 blocks, 128 channels = ~3.8M parameters (larger capacity for GPU)
NUM_BLOCKS = 6
CHANNELS = 64

model = ChessResNet(num_blocks=NUM_BLOCKS, channels=CHANNELS).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ChessResNet({NUM_BLOCKS} blocks, {CHANNELS} channels) loaded on {device}")
print(f"Total parameters: {total_params:,} | Trainable: {trainable_params:,}")

## 6. GPU Training Loop with Automatic Mixed Precision (AMP)

Features:
- **Automatic Mixed Precision (AMP)** via `torch.amp.autocast('cuda')` and `GradScaler` for 5x–10x acceleration on Tensor Cores.
- **Cosine Annealing Learning Rate** schedule with linear warmup.
- **Exponential Moving Average (EMA)** shadow weights tracking.
- **Loss Function**: Masked Cross-Entropy for Policy + Cross-Entropy for Win/Draw/Loss.

In [ ]:
import math
import time
import torch.nn.functional as F
from chesslab.inference.legal_mask import mask_logits
from chesslab.training.train_loop import ModelEMA
from chesslab.eval.predictive import evaluate_predictive

EPOCHS = 5
LR = 2e-3  # Scaled for batch size 512
WEIGHT_DECAY = 1e-4
LAM_WDL = 0.5
EVAL_EVERY_STEPS = 250
CHECKPOINT_DIR = "checkpoints/colab_gpu"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
ema = ModelEMA(model, decay=0.999)

total_steps = EPOCHS * len(train_loader)
warmup_steps = min(500, max(50, total_steps // 10))

def get_lr_factor(step: int) -> float:
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_factor)

print(f"Training settings:")
print(f"  Device: {device}")
print(f"  Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | Total steps: {total_steps:,}")
print(f"  Peak LR: {LR:.1e} | Warmup steps: {warmup_steps} | AMP: {torch.cuda.is_available()}")

best_val_top1 = 0.0
global_step = 0
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t_epoch = time.time()

    for batch_idx, batch in enumerate(train_loader):
        planes = batch["planes"].to(device, non_blocking=True)
        rule_state = batch["rule_state"].to(device, non_blocking=True)
        target_moves = batch["target_moves"].to(device, non_blocking=True)
        target_wdl = batch["target_wdl"].to(device, non_blocking=True)
        legal_masks = batch["legal_masks"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            pol_logits, wdl_logits = model(planes, rule_state)
            masked_pol = mask_logits(pol_logits, legal_masks)
            loss_pol = F.cross_entropy(masked_pol, target_moves)
            loss_wdl = F.cross_entropy(wdl_logits, target_wdl)
            total_loss = loss_pol + LAM_WDL * loss_wdl

        # Scaled backward pass
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        scheduler.step()
        ema.update(model)

        epoch_loss += total_loss.item()
        global_step += 1

        if global_step % EVAL_EVERY_STEPS == 0:
            elapsed = time.time() - t_start
            throughput = (global_step * BATCH_SIZE) / elapsed
            val_str = ""
            if val_loader:
                val_metrics = evaluate_predictive(model, val_loader, device=device, max_samples=1000)
                val_str = f" | Val Top-1: {val_metrics['top1_accuracy']*100:.2f}% | Val Top-5: {val_metrics['top5_accuracy']*100:.2f}%"
                if val_metrics["top1_accuracy"] > best_val_top1:
                    best_val_top1 = val_metrics["top1_accuracy"]
                    torch.save({
                        "step": global_step,
                        "epoch": epoch,
                        "model_state": model.state_dict(),
                        "val_top1": best_val_top1,
                    }, os.path.join(CHECKPOINT_DIR, "best_model.pt"))

            print(
                f"Step {global_step}/{total_steps} (Epoch {epoch}) | "
                f"Loss: {total_loss.item():.4f}{val_str} | "
                f"Speed: {throughput:.0f} pos/s | "
                f"LR: {scheduler.get_last_lr()[0]:.2e}"
            )

    print(f"Epoch {epoch}/{EPOCHS} completed in {time.time() - t_epoch:.1f}s")

# Save checkpoint artifacts
torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "final_model.pt"))
ema.apply_shadow(model)
torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "final_ema_model.pt"))
print(f"\nTraining complete! Checkpoints written to: {CHECKPOINT_DIR}")

## 7. Held-Out Test Set Predictive Evaluation

In [ ]:
test_shard = "data/shards/test.bin"
if os.path.exists(test_shard):
    test_dataset = BinaryChessDataset(test_shard)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=fast_collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    print(f"Evaluating EMA model on held-out test shard ({len(test_dataset):,} positions)...")
    test_metrics = evaluate_predictive(model, test_loader, device=device)
    print("\n================ HELD-OUT TEST METRICS ================")
    for k, v in test_metrics.items():
        if "accuracy" in k or "rate" in k:
            print(f"  {k:30s}: {v*100:6.2f}%")
        else:
            print(f"  {k:30s}: {v:6.4f}")
    print("=======================================================")
else:
    print(f"Test shard not found at: {test_shard}")

## 8. Tactical Puzzle Benchmark (Lichess 1k)

In [ ]:
import chess
from chesslab.inference.engine_uci import UCIEngine
from scripts.eval_puzzles import evaluate_puzzles

puzzle_csv = "data/benchmarks/puzzles_1k.csv"
if os.path.exists(puzzle_csv):
    engine = UCIEngine(model=model, device=device, search_depth=0)
    print("Evaluating model on Lichess tactical puzzles (depth=0 searchless policy)...")
    puzzle_results = evaluate_puzzles(engine, csv_path=puzzle_csv, max_puzzles=500)
    print("\n================ TACTICAL PUZZLE RESULTS ================")
    total_puzzles = puzzle_results.get('total_evaluated', puzzle_results.get('total', 0))
    print(f"  Puzzles Evaluated : {total_puzzles}")
    print(f"  Top-1 Accuracy    : {puzzle_results['top1_accuracy']*100:.2f}%")
    print(f"  Top-3 Accuracy    : {puzzle_results['top3_accuracy']*100:.2f}%")
    print(f"  Top-5 Accuracy    : {puzzle_results['top5_accuracy']*100:.2f}%")
    print("=========================================================")
else:
    print(f"Puzzle benchmark not found at: {puzzle_csv}")

## 9. Interactive Visual Move Predictor & Board Visualizer

In [ ]:
import chess
import chess.svg
from IPython.display import display, SVG
from chesslab.encode.board_planes import encode_board
from chesslab.encode.move_vocab import decode_move, encode_move

def analyze_position(fen_or_board):
    if isinstance(fen_or_board, str):
        board = chess.Board(fen_or_board)
    else:
        board = fen_or_board

    # Render SVG board visually
    display(SVG(chess.svg.board(board=board, size=350)))

    planes, rule_state = encode_board(board)
    planes_t = torch.from_numpy(planes).unsqueeze(0).to(device)
    rule_t = torch.from_numpy(rule_state).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        pol_logits, wdl_logits = model(planes_t, rule_t)
        probs = F.softmax(pol_logits, dim=-1)[0].cpu().numpy()
        wdl_probs = F.softmax(wdl_logits, dim=-1)[0].cpu().numpy()

    # Filter and rank strictly legal moves
    legal_candidates = []
    for move in board.legal_moves:
        try:
            idx = encode_move(move, board.turn)
            legal_candidates.append((move, float(probs[idx])))
        except Exception:
            continue

    legal_candidates.sort(key=lambda x: x[1], reverse=True)
    total_legal_mass = sum(p for _, p in legal_candidates)

    turn_name = 'White' if board.turn == chess.WHITE else 'Black'
    print(f"\nPosition Turn: {turn_name}")
    print(f"WDL Value Head: Win={wdl_probs[0]*100:4.1f}% | Draw={wdl_probs[1]*100:4.1f}% | Loss={wdl_probs[2]*100:4.1f}%")
    print("\nTop 5 Legal Move Recommendations:")
    for rank, (m, p) in enumerate(legal_candidates[:5], 1):
        norm_p = (p / total_legal_mass * 100) if total_legal_mass > 0 else 0.0
        san = board.san(m)
        print(f"  {rank}. {san:6s} ({m.uci():5s}) -> Policy Prob: {norm_p:5.1f}% (raw: {p*100:5.2f}%)")

# Test with standard starting position
print("Analyzing Initial Board Position:")
analyze_position(chess.Board())

# Test with Italian Game position (1.e4 e5 2.Nf3 Nc6 3.Bc4)
print("\nAnalyzing Italian Game Position:")
analyze_position("r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3")

## 10. Checkpoint Persistence & Colab Download

In [ ]:
# Helper to download trained model checkpoint to your local machine
try:
    from google.colab import files
    ckpt_file = os.path.join(CHECKPOINT_DIR, "best_model.pt")
    if os.path.exists(ckpt_file):
        print(f"Triggering download for {ckpt_file}...")
        files.download(ckpt_file)
    else:
        print(f"Checkpoint file {ckpt_file} not found.")
except ImportError:
    print(f"Local environment detected. Checkpoints are stored in: {CHECKPOINT_DIR}")